In [1]:
!pip install firebase-admin scikit-learn pandas numpy joblib

In [2]:
import firebase_admin
from firebase_admin import credentials, db as firebase_db

import pandas as pd
import numpy as np
import json, time, joblib
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [3]:
FIREBASE_DB_URL = "https://agrosmart-33c0d-default-rtdb.firebaseio.com"
SERVICE_ACCOUNT = "serviceAccountKey.json"

In [4]:
if not firebase_admin._apps:
    cred = credentials.Certificate(SERVICE_ACCOUNT)
    firebase_admin.initialize_app(cred, {"databaseURL": FIREBASE_DB_URL})

print("✅ Firebase conectado")

✅ Firebase conectado


In [5]:
RANGOS = {
    # Humedad del suelo
    "humedad_suelo": {
        "critico_bajo":  40,   # semillas necesitan mucha humedad
        "optimo_bajo":   60,
        "optimo_alto":   80,
        "critico_alto":  90,
    },
    # Temperatura ambiente
    "temperatura": {
        "critico_bajo":  10,
        "optimo_bajo":   15,
        "optimo_alto":   24,
        "critico_alto":  30,
    },
    # Humedad del aire
    "humedad_aire": {
        "critico_bajo":  40,
        "optimo_bajo":   55,
        "optimo_alto":   80,
        "critico_alto":  90,
    },
    # TDS del agua de riego (arándanos prefieren agua blanda)
    "tds": {
        "critico_bajo":  50,
        "optimo_bajo":   100,
        "optimo_alto":   400,
        "critico_alto":  600,
    },
}

In [6]:

# Estados posibles del cultivo
ESTADOS = [
    "OPTIMO",           # todo en rango ideal
    "SEQUIA",           # suelo demasiado seco
    "EXCESO_AGUA",      # suelo saturado
    "FRIO_CRITICO",     # temperatura peligrosamente baja
    "CALOR_CRITICO",    # temperatura peligrosamente alta
    "AGUA_SALINA",      # TDS muy alto, agua con demasiadas sales
    "HUMEDAD_BAJA",     # aire muy seco, riesgo de estrés hídrico
    "CONDICION_MIXTA",  # múltiples parámetros fuera de rango
]

RECOMENDACIONES = {
    "OPTIMO": (
        "🌱 Las semillas de arándano están en condiciones ideales. "
        "Mantén el riego regular y monitorea diariamente. "
        "La germinación debería ocurrir entre 2-4 semanas."
    ),
    "SEQUIA": (
        "💧 El suelo está demasiado seco para la germinación. "
        "Las semillas de arándano necesitan humedad constante. "
        "Riega de inmediato y verifica que el sustrato retenga agua. "
        "Considera cubrir con plástico transparente para retener humedad."
    ),
    "EXCESO_AGUA": (
        "🚫 Exceso de agua detectado. El encharcamiento pudre las semillas. "
        "Suspende el riego hasta que el suelo drene. "
        "Verifica que el sustrato tenga buen drenaje. "
        "Considera mezclar arena gruesa si el problema persiste."
    ),
    "FRIO_CRITICO": (
        "🌡️ Temperatura peligrosamente baja para la germinación. "
        "Las semillas de arándano no germinan por debajo de 10°C. "
        "Cubre el semillero con manta térmica o plástico esta noche. "
        "Considera trasladar a un lugar más cálido temporalmente."
    ),
    "CALOR_CRITICO": (
        "🔥 Temperatura muy alta, riesgo de daño a las semillas. "
        "Por encima de 30°C la germinación se inhibe. "
        "Proporciona sombra durante las horas más calurosas (12-16hs). "
        "Aumenta la frecuencia de riego para enfriar el sustrato."
    ),
    "AGUA_SALINA": (
        "⚠️ El agua de riego tiene demasiadas sales (TDS alto). "
        "Los arándanos son muy sensibles a la salinidad. "
        "Usa agua filtrada, de lluvia o con TDS menor a 400 ppm. "
        "El exceso de sales impide la absorción de nutrientes."
    ),
    "HUMEDAD_BAJA": (
        "💨 Humedad del aire muy baja, riesgo de desecación. "
        "Las semillas pueden perder humedad rápidamente. "
        "Cubre el semillero con film transparente perforado. "
        "Riega con aspersor fino para no desplazar las semillas."
    ),
    "CONDICION_MIXTA": (
        "⚡ Múltiples parámetros fuera del rango óptimo. "
        "Revisa humedad del suelo, temperatura y calidad del agua. "
        "Prioriza corregir la temperatura primero, luego el riego. "
        "Consulta el detalle de cada sensor en el dashboard."
    ),
}

print("✅ Rangos de arándano cargados")

✅ Rangos de arándano cargados


In [7]:
def etiquetar(row) -> str:
    """
    Clasifica el estado del cultivo según los rangos de arándano.
    Prioriza los estados más críticos primero.
    """
    hs  = row["humedad_suelo"]
    t   = row["temperatura"]
    ha  = row["humedad_aire"]
    tds = row["tds"]

    problemas = 0

    # Temperatura crítica (prioridad máxima)
    if t < RANGOS["temperatura"]["critico_bajo"]:
        return "FRIO_CRITICO"
    if t > RANGOS["temperatura"]["critico_alto"]:
        return "CALOR_CRITICO"

    # Agua salina (prioridad alta)
    if tds > RANGOS["tds"]["critico_alto"]:
        return "AGUA_SALINA"

    # Humedad del suelo
    if hs < RANGOS["humedad_suelo"]["critico_bajo"]:
        return "SEQUIA"
    if hs > RANGOS["humedad_suelo"]["critico_alto"]:
        return "EXCESO_AGUA"

    # Contar cuántos parámetros están fuera del rango óptimo
    if hs < RANGOS["humedad_suelo"]["optimo_bajo"]:
        problemas += 1
    if t < RANGOS["temperatura"]["optimo_bajo"] or \
       t > RANGOS["temperatura"]["optimo_alto"]:
        problemas += 1
    if ha < RANGOS["humedad_aire"]["critico_bajo"]:
        return "HUMEDAD_BAJA"
    if tds > RANGOS["tds"]["optimo_alto"]:
        problemas += 1

    if problemas >= 2:
        return "CONDICION_MIXTA"

    return "OPTIMO"

In [8]:
def generar_dataset(n=1000) -> pd.DataFrame:
    """
    Genera datos sintéticos realistas para semillas de arándano
    en tierra común exterior.
    """
    np.random.seed(42)

    # Distribución realista de condiciones
    datos = []
    for _ in range(n):
        # Simula variaciones naturales de clima exterior
        hora = np.random.randint(0, 24)
        es_dia = 8 <= hora <= 20

        fila = {
            "humedad_suelo": np.clip(
                np.random.normal(65, 18), 20, 100
            ),
            "temperatura": np.clip(
                np.random.normal(20 if es_dia else 14, 7), 5, 40
            ),
            "humedad_aire": np.clip(
                np.random.normal(65, 15), 20, 100
            ),
            "tds": np.clip(
                np.random.exponential(250), 30, 900
            ),
            "bomba_activa": int(np.random.random() < 0.3),
        }
        datos.append(fila)

    df = pd.DataFrame(datos)
    df["estado"] = df.apply(etiquetar, axis=1)

    print("Distribución de estados:")
    print(df["estado"].value_counts())
    return df


df = generar_dataset(1200)

Distribución de estados:
estado
OPTIMO             536
FRIO_CRITICO       200
CONDICION_MIXTA    149
EXCESO_AGUA         73
AGUA_SALINA         73
SEQUIA              70
CALOR_CRITICO       59
HUMEDAD_BAJA        40
Name: count, dtype: int64


In [9]:

FEATURES = ["humedad_suelo", "temperatura", "humedad_aire", "tds", "bomba_activa"]
TARGET   = "estado"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

modelo = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    class_weight="balanced",
)
modelo.fit(X_train_s, y_train)

y_pred = modelo.predict(X_test_s)
print("\n📈 Reporte de clasificación – Arándanos:")
print(classification_report(y_test, y_pred))

joblib.dump(modelo, "modelo_arandano.pkl")
joblib.dump(scaler, "scaler_arandano.pkl")
print("💾 Modelo guardado")



📈 Reporte de clasificación – Arándanos:
                 precision    recall  f1-score   support

    AGUA_SALINA       1.00      0.93      0.97        15
  CALOR_CRITICO       1.00      1.00      1.00        12
CONDICION_MIXTA       0.82      0.93      0.88        30
    EXCESO_AGUA       1.00      1.00      1.00        14
   FRIO_CRITICO       1.00      1.00      1.00        40
   HUMEDAD_BAJA       1.00      1.00      1.00         8
         OPTIMO       0.97      0.94      0.96       107
         SEQUIA       1.00      1.00      1.00        14

       accuracy                           0.96       240
      macro avg       0.97      0.98      0.97       240
   weighted avg       0.97      0.96      0.96       240

💾 Modelo guardado


In [10]:
# Importancia de cada sensor
importancias = pd.Series(
    modelo.feature_importances_, index=FEATURES
).sort_values(ascending=False)
print("\n🔍 Importancia de sensores:")
print(importancias)


🔍 Importancia de sensores:
temperatura      0.350245
humedad_suelo    0.319636
tds              0.178385
humedad_aire     0.149584
bomba_activa     0.002150
dtype: float64


In [11]:
def analizar_sensor(nombre, valor, rangos) -> dict:
    """Analiza un sensor individual y genera alerta específica."""
    r = rangos[nombre]

    if valor < r["critico_bajo"]:
        nivel = "CRITICO"
        emoji = "🔴"
    elif valor < r["optimo_bajo"]:
        nivel = "BAJO"
        emoji = "🟡"
    elif valor <= r["optimo_alto"]:
        nivel = "OPTIMO"
        emoji = "🟢"
    elif valor <= r["critico_alto"]:
        nivel = "ALTO"
        emoji = "🟡"
    else:
        nivel = "CRITICO"
        emoji = "🔴"

    return {"nivel": nivel, "emoji": emoji, "valor": round(valor, 1)}


def predecir_completo(sensores: dict) -> dict:
    """
    Predicción completa con estado general + análisis por sensor.
    """
    fila = pd.DataFrame([{
        "humedad_suelo": float(sensores.get("humedad_suelo", 65)),
        "temperatura":   float(sensores.get("temperatura",   20)),
        "humedad_aire":  float(sensores.get("humedad_aire",  65)),
        "tds":           float(sensores.get("tds",           200)),
        "bomba_activa":  int(sensores.get("bomba_activa",    False)),
    }])

    fila_s  = scaler.transform(fila)
    estado  = modelo.predict(fila_s)[0]
    proba   = modelo.predict_proba(fila_s)[0]
    clases  = modelo.classes_
    conf    = float(proba[list(clases).index(estado)])

    # Análisis detallado por sensor (excluyendo bomba)
    sensores_analisis = {
        k: v for k, v in RANGOS.items()
    }

    detalle = {
        sensor: analizar_sensor(
            sensor,
            fila[sensor].values[0],
            RANGOS
        )
        for sensor in RANGOS.keys()
    }

    # Puntuación de salud general (0-100)
    puntos_optimos = sum(
        1 for d in detalle.values() if d["nivel"] == "OPTIMO"
    )
    salud = int((puntos_optimos / len(detalle)) * 100)

    return {
        "estado":        estado,
        "confianza":     round(conf, 3),
        "recomendacion": RECOMENDACIONES[estado],
        "salud_general": salud,
        "detalle": {
            "humedad_suelo": detalle["humedad_suelo"],
            "temperatura":   detalle["temperatura"],
            "humedad_aire":  detalle["humedad_aire"],
            "tds":           detalle["tds"],
        },
        "timestamp": int(time.time()),
        "cultivo":   "Arándano (semilla)",
        "etapa":     "Germinación",
    }

In [ ]:

def ciclo_prediccion(intervalo_seg=15):
    """
    Lee sensores de Firebase cada N segundos,
    predice y publica el resultado.
    """
    ref_sensor = firebase_db.reference("/sensores/actual")
    ref_ml     = firebase_db.reference("/ml/prediccion_actual")

    print(f"\n🔄 Iniciando ciclo para arándanos (cada {intervalo_seg}s)…")
    print("   Presiona [Stop] en Colab para detener.\n")

    while True:
        try:
            datos = ref_sensor.get()

            if datos:
                resultado = predecir_completo(datos)
                ref_ml.set(resultado)

                ts = datetime.now().strftime("%H:%M:%S")
                emoji = "🟢" if resultado["estado"] == "OPTIMO" else "🔴"
                print(
                    f"[{ts}] {emoji} {resultado['estado']} "
                    f"| Salud: {resultado['salud_general']}% "
                    f"| Confianza: {resultado['confianza']*100:.0f}%"
                )
            else:
                print("⏳ Esperando datos del ESP32…")

        except Exception as e:
            print(f"❌ Error: {e}")

        time.sleep(intervalo_seg)


# Inicia el ciclo
ciclo_prediccion(intervalo_seg=15)


🔄 Iniciando ciclo para arándanos (cada 15s)…
   Presiona [Stop] en Colab para detener.

[01:43:42] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:43:57] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:44:12] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:44:27] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:44:42] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:44:57] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:45:13] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:45:28] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:45:43] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:45:58] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:46:13] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:46:28] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:46:44] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:46:59] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:47:14] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%
[01:47:29] 🔴 CALOR_CRITICO | Salud: 75% | Confianza: 64%